# Disease ↔ Disease Relation-Wise Merge

Merges Disease–Disease triples from Monarch, DRKG, PrimeKG, PharmKG, Hetionet,
TARKG, iBKH, and hald; resolves disease names via DO/MESH;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [6]:
import pandas as pd
import numpy as np

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/DISEASE_DISEASE/ALL_DISEASE_DISEASE.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Disease Lookup Dictionaries

In [7]:
# Disease Ontology
DO_All_File = pd.read_csv(DB_DIR + 'DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO dict: {len(DOID_disname_dict):,} entries")

DO dict: 11,804 entries


In [8]:
# MedGen: source_id → preferred name; MONDO → MESH CUI
MedGen_CUID_Source_ID = pd.read_csv(DB_DIR + 'MESH/MeSH/MedGenIDMappings.txt', sep='\t')
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id', 'source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))

MONDO_2_MESH = MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'].str.contains('MONDO', na=False)]
MONDO_2_MESH_dict    = dict(zip(MONDO_2_MESH['source_id'],     MONDO_2_MESH['#CUI_or_CN_id']))
MONDOMESH_2_des_dict = dict(zip(MONDO_2_MESH['#CUI_or_CN_id'], MONDO_2_MESH['pref_name']))
print(f"MONDO→MESH: {len(MONDO_2_MESH_dict):,} entries")

MONDO→MESH: 22,703 entries


In [9]:
# MESH combined and supplementary: ID → Name
Mesh2DOID = pd.read_csv(DB_DIR + 'DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv', sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))

MESH = pd.read_csv(DB_DIR + 'MESH/MESH_Combined.csv')
MESH_dict = dict(zip(MESH['ID'], MESH['Name']))

Mesh_supp = pd.read_csv(DB_DIR + 'MESH/Mesh_Supplementary_Info.csv')
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))

print(f"MESH dict: {len(MESH_dict):,} | MESH supp: {len(Mesh_supp_dict):,}")

MESH dict: 38,520 | MESH supp: 324,150


## 2. Helper — Resolve Disease Node (head or tail)

In [10]:
def resolve_disease_node(df, node):
    """Map MONDO→MESH, derive detail_name, assign id_is for head or tail."""
    df[node] = df[node].map(MONDO_2_MESH_dict).fillna(df[node])
    df = df[~df[node].astype(str).str.startswith('MONDO')].copy()
    df[f'{node}_detail_name'] = df[node].apply(
        lambda x: DOID_disname_dict.get(x)
                  if isinstance(x, str) and x.startswith('DOID')
                  else (MESH_dict.get(x) or Mesh_supp_dict.get(x) or MONDOMESH_2_des_dict.get(x))
    )
    df[f'{node}_id_is'] = np.where(
        df[node].isna(), None,
        np.where(df[node].astype(str).str.startswith('DOID'), 'DOID', 'MESH')
    )
    return df

## 3. Load KG Sources

### 3a. Monarch

In [11]:
Monarch_Disease_Disease = pd.read_csv(PROC_DIR + 'Monarch/Human/Human_Disease_Disease_Monarch.csv')
Monarch_Disease_Disease.columns = Monarch_Disease_Disease.columns.str.lower()
Monarch_Disease_Disease.rename(columns={'kgsource': 'kg_source', 'head_name': 'head_detail_name', 'tail_name': 'tail_detail_name'}, inplace=True)
Monarch_Disease_Disease = resolve_disease_node(Monarch_Disease_Disease, 'head')
Monarch_Disease_Disease = resolve_disease_node(Monarch_Disease_Disease, 'tail')
print(f"Monarch:  {len(Monarch_Disease_Disease):,} rows")

Monarch_Disease_Disease['kg_type'] = 'Generalised'
Monarch_Disease_Disease['species'] = 'Homo species'

Monarch_Disease_Disease.head(2)

Monarch:  35,048 rows


,head,tail,relation_type,relation_source,kg_source,head_detail_name,head_type,head_id_is,head_taxon,head_taxon_label,...,head_taxon_name,tail_taxon_name,head_type_clean,tail_type_clean,relation,species_species,head_do_name,tail_do_name,kg_type,species
0,C3272833,DOID:8632,subclass_of,infores:mondo,MonarchKG,Colorectal Kaposi sarcoma,Disease,MESH,NaN,NaN,...,Homo sapiens,Homo sapiens,Disease,Disease,Disease_Disease,NaN,MONDO:0024659,MONDO:0005055,Generalised,Homo species
1,C3272833,DOID:10155,subclass_of,infores:mondo,MonarchKG,Colorectal Kaposi sarcoma,Disease,MESH,NaN,NaN,...,Homo sapiens,Homo sapiens,Disease,Disease,Disease_Disease,NaN,MONDO:0024659,MONDO:0005814,Generalised,Homo species


### 3b. DRKG

In [12]:
DRKG_Disease_Disease = pd.read_csv(PROC_DIR + 'DRKG/DRKG_Disease_Disease.csv')
DRKG_Disease_Disease.columns = DRKG_Disease_Disease.columns.str.lower()
DRKG_Disease_Disease.rename(columns={'tail_name': 'tail_detail_name'}, inplace=True)
print(f"DRKG:     {len(DRKG_Disease_Disease):,} rows")

DRKG_Disease_Disease['kg_type'] = 'Generalised'
DRKG_Disease_Disease['kg_source'] = 'DRKG'
DRKG_Disease_Disease['species'] = 'Homo species'

DRKG_Disease_Disease['head_id_is'] = np.where(DRKG_Disease_Disease['head'].str.startswith('DOID'), 'DO_ID', 'MESH')

DRKG_Disease_Disease

DRKG:     535 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type,species
0,D009373,Disease_Disease,DOID:11934,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,"Neoplasms, Germ Cell and Embryonal",head and neck cancer,MESH,DOID,Disease::MESH:D009373,Disease::DOID:11934,Generalised,Homo species
1,D015179,Disease_Disease,DOID:3571,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Colorectal Neoplasms,liver cancer,MESH,DOID,Disease::MESH:D015179,Disease::DOID:3571,Generalised,Homo species
2,D007680,Disease_Disease,DOID:4045,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Kidney Neoplasms,muscle cancer,MESH,DOID,Disease::MESH:D007680,Disease::DOID:4045,Generalised,Homo species
3,D005185,Disease_Disease,DOID:13223,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Fallopian Tube Neoplasms,uterine fibroid,MESH,DOID,Disease::MESH:D005185,Disease::DOID:13223,Generalised,Homo species
4,D001859,Disease_Disease,DOID:10283,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Bone Neoplasms,prostate cancer,MESH,DOID,Disease::MESH:D001859,Disease::DOID:10283,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
530,D015179,Disease_Disease,DOID:8577,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Colorectal Neoplasms,ulcerative colitis,MESH,DOID,Disease::MESH:D015179,Disease::DOID:8577,Generalised,Homo species
531,D009373,Disease_Disease,DOID:13499,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,"Neoplasms, Germ Cell and Embryonal",jejunal cancer,MESH,DOID,Disease::MESH:D009373,Disease::DOID:13499,Generalised,Homo species
532,D010190,Disease_Disease,DOID:10534,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Pancreatic Neoplasms,stomach cancer,MESH,DOID,Disease::MESH:D010190,Disease::DOID:10534,Generalised,Homo species
533,D015179,Disease_Disease,DOID:3121,Disease,Hetionet::DrD::Disease:Disease,Disease,DRKG,Colorectal Neoplasms,gallbladder cancer,MESH,DOID,Disease::MESH:D015179,Disease::DOID:3121,Generalised,Homo species


### 3c. PrimeKG

In [13]:
PrimeKG_Disease_Disease = pd.read_csv(PROC_DIR + 'PrimeKG/PrimeKG_Disease_Disease.csv')
PrimeKG_Disease_Disease.columns = PrimeKG_Disease_Disease.columns.str.lower()
print(f"PrimeKG:  {len(PrimeKG_Disease_Disease):,} rows")


PrimeKG_Disease_Disease['kg_type'] = 'Generalised'
PrimeKG_Disease_Disease['kg_source'] = 'PrimeKG'
PrimeKG_Disease_Disease['species'] = 'Homo species'

PrimeKG_Disease_Disease.head(2)

PrimeKG:  5,316 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,DOID:0080637,Disease_Disease,DOID:10629,Disease,parent-child,Disease,PrimeKG,DOID_ID,DOID_ID,isolated microphthalmia,microphthalmia,Generalised,Homo species
1,DOID:13564,Disease_Disease,DOID:0050153,Disease,parent-child,Disease,PrimeKG,DOID_ID,DOID_ID,aspergillosis,pulmonary aspergilloma,Generalised,Homo species


### 3d. PharmKG

In [14]:
PharmKG_Disease_Disease = pd.read_csv(PROC_DIR + 'PharmKG/PharmKG_Disease_Disease.csv')
PharmKG_Disease_Disease.columns = PharmKG_Disease_Disease.columns.str.lower()
PharmKG_Disease_Disease.rename(columns={'head_orignal': 'head_detail_name', 'tail_orignal': 'tail_detail_name'}, inplace=True)
print(f"PharmKG:  {len(PharmKG_Disease_Disease):,} rows")

PharmKG_Disease_Disease['kg_type'] = 'Generalised'
PharmKG_Disease_Disease['kg_source'] = 'PharmKG'
PharmKG_Disease_Disease['species'] = 'Homo species'

PharmKG_Disease_Disease.head(2)

PharmKG:  46 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,pubmed_id,sentence_tokenized,kg_type,species
0,DOID:2121,Disease_Disease,DOID:9821,disease,J,disease,PharmKG,DOID_ID,DOID_ID,ectodermal dysplasia,choroideremia,"21067479.0, 21067479.0",'CONCLUSION : An X-autosome chromosomal transl...,Generalised,Homo species
1,DOID:10017,Disease_Disease,DOID:13543,disease,Te,disease,PharmKG,DOID_ID,DOID_ID,multiple endocrine neoplasia type 1,hyperparathyroidism,"10856877.0, 7491535.0, 1362402.0, 9854574.0, 2...",'Germ line mutations of the multiple_endocrine...,Generalised,Homo species


### 3e. Hetionet

In [15]:
Hetionet_Disease_Disease = pd.read_csv(PROC_DIR + 'Hetionet/Disease_Disease_Hetionet.csv')
Hetionet_Disease_Disease.columns = Hetionet_Disease_Disease.columns.str.lower()
Hetionet_Disease_Disease.rename(columns={'head_name': 'head_detail_name', 'tail_name': 'tail_detail_name'}, inplace=True)
print(f"Hetionet: {len(Hetionet_Disease_Disease):,} rows")

Hetionet_Disease_Disease['kg_type'] = 'Generalised'
Hetionet_Disease_Disease['species'] = 'Homo species'

Hetionet_Disease_Disease.head(2)

Hetionet: 543 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,DOID:2994,Disease_Disease,DOID:11934,Disease,Disease–resembles–Disease,Disease,Hetionet,DOID,DOID,germ cell cancer,head and neck cancer,Generalised,Homo species
1,DOID:219,Disease_Disease,DOID:3571,Disease,Disease–resembles–Disease,Disease,Hetionet,DOID,DOID,colon cancer,liver cancer,Generalised,Homo species


### 3f. TARKG

In [16]:
TARKG_Disease_Disease = pd.read_csv(PROC_DIR + 'TARKG/Disease_Disease_TARKG.csv')
TARKG_Disease_Disease.columns = TARKG_Disease_Disease.columns.str.lower()
print(f"TARKG:    {len(TARKG_Disease_Disease):,} rows")

TARKG_Disease_Disease['kg_type'] = 'Generalised'
TARKG_Disease_Disease['species'] = 'Homo species'

TARKG_Disease_Disease.head(2)

TARKG:    35,083 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,DOID:0001816,Disease_Disease,DOID:175,Disease,is a,Disease,TARKG,angiosarcoma,vascular cancer,DOID,DOID,addKG-1,addKG,0,Generalised,Homo species
1,DOID:0001816,Disease_Disease,DOID:175,Disease,IS_A,Disease,TARKG,angiosarcoma,vascular cancer,DOID,DOID,OpenBioLink-33345,OpenBioLink,0,Generalised,Homo species


### 3g. iBKH

In [17]:
iBKH_Disease_Disease = pd.read_csv(PROC_DIR + 'iBKH/Disease_Disease_ibkh.csv')
iBKH_Disease_Disease.columns = iBKH_Disease_Disease.columns.str.lower()
print(f"iBKH:     {len(iBKH_Disease_Disease):,} rows")

iBKH_Disease_Disease['kg_type'] = 'Generalised'
iBKH_Disease_Disease['species'] = 'Homo species'


iBKH_Disease_Disease.head(2)

iBKH:     11,022 rows


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,DOID:0001816,Disease_Disease,DOID:175,Disease,Disease,iBKH,angiosarcoma,vascular cancer,DOID,DOID,Generalised,Homo species
1,DOID:175,Disease_Disease,DOID:176,Disease,Disease,iBKH,vascular cancer,cardiovascular cancer,DOID,DOID,Generalised,Homo species


### 3h. hald

In [18]:
hald = pd.read_csv(PROC_DIR + 'hald/Disease_Disease_HALD.csv')
hald.rename(columns={'relation_source': 'kg_source'}, inplace=True)
hald.columns = hald.columns.str.lower()
print(f"hald:   {len(hald):,} rows")

hald['kg_type'] = 'Aging'
hald['species'] = 'Homo species'


hald.head(2)

hald:   80,162 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,D029424,Disease_Disease,D007249,Disease,defined,Disease,HALD_KG,"Pulmonary Disease, Chronic Obstructive",Inflammation,MESH,MESH,Aging,Homo species
1,D000855,Disease_Disease,D055948,Disease,associate,Disease,HALD_KG,Anorexia,Sarcopenia,MESH,MESH,Aging,Homo species


### 3g. pheknowlator

In [19]:
pheknowlator = pd.read_csv(PROC_DIR + 'pheknowlator/Disease_Disease.csv')
pheknowlator.rename(columns={'relation_source': 'kg_source'}, inplace=True)
pheknowlator.columns = pheknowlator.columns.str.lower()
print(f"hald:   {len(hald):,} rows")

pheknowlator['kg_type'] = 'Generalised'
pheknowlator['species'] = 'Homo species'

pheknowlator.head(2)

hald:   80,162 rows


,head,relation,tail,head_type,tail_type,head_detail_name,head_detail_name.1,tail_detail_name,head_id_is,tail_id_is,kg_source,kg_type,species
0,DOID:61,Disease_Disease,DOID:61,Disease,Disease,mitral valve disease,mitral valve disease,mitral valve disease,DOID,DOID,pheknowlator,Generalised,Homo species
1,DOID:0050948,Disease_Disease,DOID:0050948,Disease,Disease,autosomal dominant hypophosphatemic rickets,autosomal dominant hypophosphatemic rickets,autosomal dominant hypophosphatemic rickets,DOID,DOID,pheknowlator,Generalised,Homo species


## 4. Check and Fix Duplicate Columns

In [20]:
SOURCE_DFS = [
    ('Monarch_Disease_Disease',  Monarch_Disease_Disease),
    ('DRKG_Disease_Disease',     DRKG_Disease_Disease),
    ('PrimeKG_Disease_Disease',  PrimeKG_Disease_Disease),
    ('PharmKG_Disease_Disease',  PharmKG_Disease_Disease),
    ('Hetionet_Disease_Disease', Hetionet_Disease_Disease),
    ('TARKG_Disease_Disease',    TARKG_Disease_Disease),
    ('iBKH_Disease_Disease',     iBKH_Disease_Disease),
    ('hald',                   hald),
    ('pheknowlator',  pheknowlator)
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[Monarch_Disease_Disease] ✓ no duplicates
[DRKG_Disease_Disease] ✓ no duplicates
[PrimeKG_Disease_Disease] ✓ no duplicates
[PharmKG_Disease_Disease] ✓ no duplicates
[Hetionet_Disease_Disease] ✓ no duplicates
[TARKG_Disease_Disease] ✓ no duplicates
[iBKH_Disease_Disease] ✓ no duplicates
[hald] ✓ no duplicates
[pheknowlator] ✓ no duplicates


In [21]:
# Monarch_Disease_Disease  = Monarch_Disease_Disease.loc[:,  ~Monarch_Disease_Disease.columns.duplicated(keep='first')]
# DRKG_Disease_Disease     = DRKG_Disease_Disease.loc[:,     ~DRKG_Disease_Disease.columns.duplicated(keep='first')]
# PrimeKG_Disease_Disease  = PrimeKG_Disease_Disease.loc[:,  ~PrimeKG_Disease_Disease.columns.duplicated(keep='first')]
# PharmKG_Disease_Disease  = PharmKG_Disease_Disease.loc[:,  ~PharmKG_Disease_Disease.columns.duplicated(keep='first')]
# Hetionet_Disease_Disease = Hetionet_Disease_Disease.loc[:, ~Hetionet_Disease_Disease.columns.duplicated(keep='first')]
# TARKG_Disease_Disease    = TARKG_Disease_Disease.loc[:,    ~TARKG_Disease_Disease.columns.duplicated(keep='first')]
# iBKH_Disease_Disease     = iBKH_Disease_Disease.loc[:,     ~iBKH_Disease_Disease.columns.duplicated(keep='first')]
# hald                   = hald.loc[:,                   ~hald.columns.duplicated(keep='first')]

# for name, df in SOURCE_DFS:
#     remaining = df.columns[df.columns.duplicated()].tolist()
#     print(f"[{name}] {'✓ clean' if not remaining else '✗ dupes: ' + str(remaining)}")

## 5. Align Schemas and Concatenate

In [22]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 170,320 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,C3272833,Disease_Disease,DOID:8632,Disease,subclass_of,Disease,MonarchKG,MESH,DOID,Colorectal Kaposi sarcoma,Kaposi's sarcoma,Generalised,Homo species
1,C3272833,Disease_Disease,DOID:10155,Disease,subclass_of,Disease,MonarchKG,MESH,DOID,Colorectal Kaposi sarcoma,intestinal cancer,Generalised,Homo species


## 6. Standardise Fixed-Value Columns

In [23]:
final_df['head_type'] = 'Disease'
final_df['tail_type'] = 'Disease'
# Normalise DOID_ID → DOID artefact from some sources
final_df['head_id_is'] = final_df['head_id_is'].str.replace('DOID_ID', 'DOID', regex=False)
final_df['tail_id_is'] = final_df['tail_id_is'].str.replace('DOID_ID', 'DOID', regex=False)

print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))

Unique relation: {'Disease_Disease'}
Unique head_id_is: {'DO_ID', 'MESH', 'DOID'}
Unique tail_id_is: {'DOID', 'MESH'}


## 7. Deduplicate and Add Schema Columns

In [24]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

After deduplication: 94,317 rows
Final shape: (94317, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,C0000809,Disease_Disease,C0178829,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Habitual spontaneous abortion,Reproductive system disorder,Generalised,Homo species
1,C0000833,Disease_Disease,C0009450,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Abscess,Infectious disease,Generalised,Homo species
2,C0000880,Disease_Disease,C0014238,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Acanthamoeba keratitis,Parasitic endophthalmitis,Generalised,Homo species
3,C0000880,Disease_Disease,C0729777,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Acanthamoeba keratitis,Corneal infection,Generalised,Homo species
4,C0000880,Disease_Disease,CN281745,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Acanthamoeba keratitis,Protozoa infectious disease,Generalised,Homo species


## 8. QC — NaN Check

In [25]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,2597,0,2597
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


In [26]:
final_df_dedup[final_df_dedup['head_id_is'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


## 9. Save Output

In [27]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 94,317 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_DISEASE/ALL_DISEASE_DISEASE.csv


In [28]:
#

# Final MAppping of disease

In [29]:
# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (df[detail_col].astype(str).str.lower().str.strip().map(mapping_dict))

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),id_is_col] = 'DOID'
    
    # everything else -> MESH
    df.loc[~df[id_col].str.startswith('DOID:', na=False)&df[id_col].notna(),id_is_col] = 'MESH'

    return df

In [30]:
final_file = pd.read_csv(OUT_PATH)

final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='head')
final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='tail')

final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,C0000809,Disease_Disease,C0178829,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Habitual spontaneous abortion,Reproductive system disorder,Generalised,Homo species
1,C0000833,Disease_Disease,C0009450,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Abscess,Infectious disease,Generalised,Homo species
2,C0000880,Disease_Disease,C0014238,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Acanthamoeba keratitis,Parasitic endophthalmitis,Generalised,Homo species
3,C0000880,Disease_Disease,C0729777,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Acanthamoeba keratitis,Corneal infection,Generalised,Homo species
4,C0000880,Disease_Disease,CN281745,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Acanthamoeba keratitis,Protozoa infectious disease,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
94312,DOID:9993,Disease_Disease,DOID:9406,Disease,considered,Disease,HALD_KG,DOID,DOID,Hypoglycemia,Hypopituitarism,Aging,Homo species
94313,DOID:9993,Disease_Disease,DOID:9970,Disease,associate,Disease,HALD_KG,DOID,DOID,Hypoglycemia,Obesity,Aging,Homo species
94314,DOID:9997,Disease_Disease,C0151864,Disease,subclass_of,Disease,MonarchKG,DOID,MESH,peripartum cardiomyopathy,Pregnancy disorder,Generalised,Homo species
94315,DOID:9997,Disease_Disease,C5681849,Disease,subclass_of,Disease,MonarchKG,DOID,MESH,peripartum cardiomyopathy,Non-familial dilated cardiomyopathy,Generalised,Homo species


In [31]:
final_file[final_file['head'].isna()], final_file[final_file['tail'].isna()],

(Empty DataFrame
 Columns: [head, relation, tail, head_type, relation_type, tail_type, kg_source, head_id_is, tail_id_is, head_detail_name, tail_detail_name, kg_type, species]
 Index: [],
 Empty DataFrame
 Columns: [head, relation, tail, head_type, relation_type, tail_type, kg_source, head_id_is, tail_id_is, head_detail_name, tail_detail_name, kg_type, species]
 Index: [])

In [32]:
final_file.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_file):,} rows → {OUT_PATH}")

Saved 94,317 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_DISEASE/ALL_DISEASE_DISEASE.csv


In [33]:
f = pd.read_csv('/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/DISEASE_DISEASE/ALL_DISEASE_DISEASE.csv')
f

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,C0000809,Disease_Disease,C0178829,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Habitual spontaneous abortion,Reproductive system disorder,Generalised,Homo species
1,C0000833,Disease_Disease,C0009450,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Abscess,Infectious disease,Generalised,Homo species
2,C0000880,Disease_Disease,C0014238,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Acanthamoeba keratitis,Parasitic endophthalmitis,Generalised,Homo species
3,C0000880,Disease_Disease,C0729777,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Acanthamoeba keratitis,Corneal infection,Generalised,Homo species
4,C0000880,Disease_Disease,CN281745,Disease,subclass_of,Disease,MonarchKG,MESH,MESH,Acanthamoeba keratitis,Protozoa infectious disease,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
94312,DOID:9993,Disease_Disease,DOID:9406,Disease,considered,Disease,HALD_KG,DOID,DOID,Hypoglycemia,Hypopituitarism,Aging,Homo species
94313,DOID:9993,Disease_Disease,DOID:9970,Disease,associate,Disease,HALD_KG,DOID,DOID,Hypoglycemia,Obesity,Aging,Homo species
94314,DOID:9997,Disease_Disease,C0151864,Disease,subclass_of,Disease,MonarchKG,DOID,MESH,peripartum cardiomyopathy,Pregnancy disorder,Generalised,Homo species
94315,DOID:9997,Disease_Disease,C5681849,Disease,subclass_of,Disease,MonarchKG,DOID,MESH,peripartum cardiomyopathy,Non-familial dilated cardiomyopathy,Generalised,Homo species


In [35]:
set(f['kg_type'])

{'Aging', 'Aging::Generalised', 'Generalised'}